# BigBasket Category Performance Diagnostic — Part 4

Independent Pandas cleaning, analysis, visualization, and cross-validation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
orders = pd.read_csv('orders_raw.csv')
products = pd.read_csv('products.csv')
print('Raw shape:', orders.shape)
orders.info()
display(orders.describe(include='all'))
print(orders['status'].value_counts())

### Initial observations
The raw export has more than 500 rows, mixed city/category casing and whitespace, missing `amount_inr`, and unusually large revenue values. Null ratings for Cancelled/Pending are expected because those orders are not rated.

In [ ]:
before = len(orders)
df = orders.drop_duplicates(subset='order_id', keep='first').copy()
print('Rows before:', before)
print('Duplicate rows removed:', before-len(df))
print('Rows after:', len(df))

In [ ]:
df['city'] = df['city'].str.strip().str.title()
df['category'] = df['category'].str.strip().str.title()
print('Cities:', sorted(df['city'].unique()))
print('Categories:', sorted(df['category'].unique()))

In [ ]:
df['amount_inr'] = pd.to_numeric(df['amount_inr'], errors='coerce')
print('Missing amount_inr:', df['amount_inr'].isna().sum())
print('Null ratings by status:')
print(df.groupby('status')['rating'].apply(lambda s: s.isna().sum()))
# Missing revenue is excluded from revenue calculations; rating nulls are intentionally left unchanged.

In [ ]:
delivered = df[(df['status']=='Delivered') & df['amount_inr'].notna()]
Q1 = delivered['amount_inr'].quantile(0.25)
Q3 = delivered['amount_inr'].quantile(0.75)
IQR = Q3-Q1
upper_fence = Q3 + 1.5*IQR
print(f'Q1={Q1}, Q3={Q3}, IQR={IQR}, upper fence={upper_fence}')
print('Rows capped:', (delivered['amount_inr']>upper_fence).sum())
df['amount_capped'] = df['amount_inr']
df.loc[delivered.index,'amount_capped'] = df.loc[delivered.index,'amount_inr'].clip(upper=upper_fence)

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['month'] = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.month_name()
df['revenue_per_unit'] = df['amount_capped']/df['quantity']
df['is_delivered'] = df['status'].eq('Delivered')

In [ ]:
revenue_df = df[df['is_delivered'] & df['amount_capped'].notna()].copy()
category_revenue = revenue_df.groupby('category')['amount_capped'].sum().sort_values(ascending=False)
print(category_revenue)
top_category = category_revenue.index[0]
print('Top category:', top_category)

In [ ]:
merged = df.merge(products, on='product_id', how='left', suffixes=('', '_product'))
supplier_revenue = merged[merged['is_delivered'] & merged['amount_capped'].notna()].groupby('supplier')['amount_capped'].sum().sort_values(ascending=False)
print(supplier_revenue)
top_supplier = supplier_revenue.index[0]
print('Top supplier:', top_supplier)
print('Top category matches Part 1:', top_category == 'Household Essentials')
print('Top supplier matches Part 1:', top_supplier == 'HomeEssentials Traders')

In [ ]:
category_revenue.plot(kind='bar')
plt.title('Household Essentials is the highest-revenue category after cleaning')
plt.xlabel('Category'); plt.ylabel('Revenue (INR)'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
monthly = revenue_df.groupby('month')['amount_capped'].sum()
monthly.plot(kind='line', marker='o')
plt.title('May 2026 has the highest cleaned delivered revenue')
plt.xlabel('Month number'); plt.ylabel('Revenue (INR)'); plt.tight_layout(); plt.show()

In [ ]:
supplier_revenue.plot(kind='bar')
plt.title('HomeEssentials Traders generates the highest supplier revenue')
plt.xlabel('Supplier'); plt.ylabel('Revenue (INR)'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## Exactly 3 What / Why it matters / Next step observations

**1. What:** Household Essentials is the top category after cleaning, at INR 20,910.  
**Why it matters:** It remains the strongest category after the required data-quality treatment.  
**Next step:** Review the products driving this category's revenue and protect their availability.

**2. What:** May 2026 has the highest cleaned delivered revenue among the six months.  
**Why it matters:** Revenue is uneven across months, making the monthly peak useful for comparing category mix.  
**Next step:** Compare May's category mix with the weaker months.

**3. What:** HomeEssentials Traders is the top supplier at INR 20,910 after cleaning.  
**Why it matters:** The supplier is directly associated with the strongest category-level revenue result.  
**Next step:** Review the individual products supplied by HomeEssentials Traders to understand their contribution.

**Cross-validation:** The top category is Household Essentials and the top supplier is HomeEssentials Traders, matching Part 1. Exact totals differ because Part 4 removes duplicates, excludes missing revenue, and caps IQR outliers.